# PyADM1ODE: Basic Digester Example

This notebook demonstrates the simplest possible PyADM1 configuration: a single digester with substrate feed and integrated gas storage.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/PyADM1ODE/blob/master/examples/colab_01_basic_digester.ipynb)

## 1. Setup Environment

We need to install the necessary dependencies, including Mono (for .NET compatibility) and the `pyadm1` package.

In [ ]:
# Install Mono and other dependencies
!apt-get update
!apt-get install -y mono-devel

# Clone the repository and install pyadm1
!git clone https://github.com/dgaida/PyADM1ODE.git
%cd PyADM1ODE
!pip install -e .

## 2. Import Libraries

In [ ]:
from pathlib import Path
from pyadm1.configurator.plant_builder import BiogasPlant
from pyadm1.substrates.feedstock import Feedstock
from pyadm1.core.adm1 import get_state_zero_from_initial_state
from pyadm1.configurator.plant_configurator import PlantConfigurator

## 3. Run Simulation

In [ ]:
# Setup paths
data_path = Path("data/initial_states")
initial_state_file = data_path / "digester_initial8.csv"

# 1. Create feedstock
feeding_freq = 48
feedstock = Feedstock(feeding_freq=feeding_freq)

# 2. Load initial state from CSV
adm1_state = get_state_zero_from_initial_state(str(initial_state_file))

# 3. Create biogas plant
plant = BiogasPlant("Quickstart Plant")
configurator = PlantConfigurator(plant, feedstock)

# 4. Define substrate feed (Corn silage: 15 m³/d, Manure: 10 m³/d)
Q_substrates = [15, 10, 0, 0, 0, 0, 0, 0, 0, 0]

# 5. Add digester
configurator.add_digester(
    digester_id="main_digester",
    V_liq=2000.0,
    V_gas=300.0,
    T_ad=308.15,  # 35°C
    name="Main Digester",
    load_initial_state=True,
    initial_state_file=str(initial_state_file),
    Q_substrates=Q_substrates,
)

# 6. Initialize and simulate
plant.initialize()
results = plant.simulate(duration=5.0, dt=1.0/24.0, save_interval=1.0)

print("\nSimulation completed successfully!")

## 4. Analyze Results

In [ ]:
for result in results:
    time = result["time"]
    comp_results = result["components"]["main_digester"]
    print(f"Day {time:.1f}:")
    print(f"  Biogas:  {comp_results.get('Q_gas', 0):>8.1f} m³/d")
    print(f"  Methane: {comp_results.get('Q_ch4', 0):>8.1f} m³/d")
    print(f"  pH:      {comp_results.get('pH', 0):>8.2f}")